# ORBIT Seed — continue training on a Kaggle GPU

Attach the private Kaggle Dataset containing orbit-seed-gpu-ready.zip, then enable **Settings → Accelerator → GPU**. This notebook continues the existing checkpoint; it does not initialize a new model.

The first run defaults to 100 additional updates as a GPU smoke test. The model's own evaluation notes warn that the current synthetic corpus is too small to produce coherent general conversation, so more steps on the same text are not a substitute for better training data.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. In Notebook Settings, set Accelerator to GPU, then restart the session.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
import zipfile

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
archives = [p for p in INPUT.rglob('*.zip') if 'orbit-seed-gpu-ready' in p.name.lower()]
if not archives:
    raise FileNotFoundError('Attach the Kaggle Dataset containing orbit-seed-gpu-ready.zip using Add Input.')
archive = archives[0]
target_root = WORK / 'orbit_seed_unpack'
target_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as zf:
    root = target_root.resolve()
    for member in zf.infolist():
        destination = (target_root / member.filename).resolve()
        if destination != root and root not in destination.parents:
            raise RuntimeError(f'Unsafe path in archive: {member.filename}')
    zf.extractall(target_root)

configs = list(target_root.rglob('config.json'))
projects = [p.parent for p in configs if (p.parent / 'model.safetensors').exists() and (p.parent / 'resume_train_gpu.py').exists()]
if not projects:
    raise FileNotFoundError('Could not locate ORBIT config, weights and GPU script in the archive.')
ORBIT_DIR = projects[0]
print('ORBIT project:', ORBIT_DIR)


In [ ]:
%pip -q install sentencepiece safetensors


In [ ]:
import subprocess, sys

EXTRA_STEPS = 100   # Change this for a longer continuation run
BATCH_SIZE = 32
SEQUENCE_LENGTH = 160

command = [sys.executable, str(ORBIT_DIR / 'resume_train_gpu.py'),
           '--device', 'cuda', '--steps', str(EXTRA_STEPS),
           '--batch-size', str(BATCH_SIZE), '--seq-len', str(SEQUENCE_LENGTH)]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)


In [ ]:
import shutil

OUTPUT_ZIP = WORK / 'orbit-seed-gpu-checkpoint.zip'
shutil.make_archive(str(OUTPUT_ZIP.with_suffix('')), 'zip', root_dir=ORBIT_DIR)
print('Updated checkpoint:', OUTPUT_ZIP)
print('Download this ZIP from the Kaggle notebook Output / Files panel.')
